In [1]:
from pathlib import Path
import random

import numpy as np
from PIL import Image
from quickdraw import QuickDrawDataGroup
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader

In [2]:
SEED = 42
SAMPLES_PER_CATEGORY = 3000
IMAGE_SIZE = 128
EMBEDDING_SIZE = 128

CATEGORY_FILE = Path("../../data/categories/objects.txt")
MODEL_PATH = Path("../../data/models/objects_cnn_supcon_model.pt")
CENTROID_DIR = Path("../../data/centroids")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

mps


In [4]:
with open(CATEGORY_FILE) as f:
    categories = [
        x.strip()
        for x in f.read().replace("\n", ",").split(",")
        if x.strip()
    ]

label_map = {
    name: i
    for i, name in enumerate(categories)
}


print(categories)

['backpack', 'bed', 'book', 'chair', 'clock', 'cup', 'door', 'key', 'knife', 'laptop', 'shoe', 'spoon', 'table', 'television', 'toothbrush']


In [5]:
class SupConObjectCNN(nn.Module):

    def __init__(self, num_classes, embedding_size=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten()
        )

        self.embedding = nn.Sequential(
            nn.Linear(32 * 32 * 32, embedding_size),
            nn.ReLU()
        )

        self.classifier = nn.Linear(
            embedding_size,
            num_classes
        )

    def get_embedding(self, x):
        return self.embedding(
            self.features(x)
        )

    def forward(self, x):
        embedding = self.get_embedding(x)
        logits = self.classifier(embedding)
        return logits, embedding


model = SupConObjectCNN(
    len(categories),
    EMBEDDING_SIZE
).to(device)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Loaded:", MODEL_PATH)

Loaded: ../../data/models/objects_cnn_supcon_model.pt


In [7]:
X = []
y = []

for class_idx, category in enumerate(categories):

    print("Loading:", category)

    group = QuickDrawDataGroup(
        category,
        max_drawings=SAMPLES_PER_CATEGORY,
        print_messages=False
    )

    count = 0

    for drawing in group.drawings:
    
        image = drawing.image.convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.LANCZOS
        )
    
        array = np.array(
            image,
            dtype=np.uint8
        )
    
        array = 255 - array
    
        X.append(array)
        y.append(class_idx)
    
        count += 1
    
        if count >= SAMPLES_PER_CATEGORY:
            break

X = np.stack(X)
y = np.array(y)

all_indices = np.arange(len(y))

train_indices, _ = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Training drawings:", len(train_indices))

Loading: backpack
Loading: bed
Loading: book
Loading: chair
Loading: clock
Loading: cup
Loading: door
Loading: key
Loading: knife
Loading: laptop
Loading: shoe
Loading: spoon
Loading: table
Loading: television
Loading: toothbrush
Training drawings: 36000


In [8]:
class EmbeddingDataset(Dataset):

    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        real_idx = self.indices[idx]

        image = (
            self.X[real_idx].astype(np.float32)
            / 255.0
        )

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        return image, int(self.y[real_idx])


loader = DataLoader(
    EmbeddingDataset(
        X,
        y,
        train_indices
    ),
    batch_size=64,
    shuffle=False
)

train_embeddings = []
train_labels = []

model.eval()

with torch.no_grad():

    for images, labels in loader:

        images = images.to(device)

        _, embeddings = model(images)

        train_embeddings.append(
            embeddings.cpu()
        )

        train_labels.append(labels)

train_embeddings = torch.cat(
    train_embeddings
)

train_labels = torch.cat(
    train_labels
)

print(train_embeddings.shape)

torch.Size([36000, 128])


In [9]:
def compute_centroids(k):

    centroids = {}

    for class_idx, class_name in enumerate(categories):

        class_embeddings = train_embeddings[
            train_labels == class_idx
        ]

        # Overall class mean
        class_mean = class_embeddings.mean(
            dim=0,
            keepdim=True
        )

        # Find embeddings closest to mean
        similarities = (
            F.normalize(class_embeddings, dim=1)
            @ F.normalize(class_mean, dim=1).T
        ).squeeze(1)

        top_k_indices = torch.topk(
            similarities,
            k=k
        ).indices

        # Average k representative embeddings
        centroid = class_embeddings[
            top_k_indices
        ].mean(
            dim=0,
            keepdim=True
        )

        # Normalize for cosine scoring
        centroid = F.normalize(
            centroid,
            dim=1
        )

        centroids[class_name] = (
            centroid.numpy()
            .astype(np.float32)
        )

    return centroids
    

In [12]:
CENTROID_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for k in [1, 5, 10]:

    centroids = compute_centroids(k)

    save_path = (
        CENTROID_DIR
        / f"objects_{k}.npz"
    )

    np.savez(
        save_path,
        **centroids
    )

    print(
        f"Saved {save_path} "
    )

Saved ../../data/centroids/objects_1.npz 
Saved ../../data/centroids/objects_5.npz 
Saved ../../data/centroids/objects_10.npz 


In [13]:
_, test_indices = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

test_loader = DataLoader(
    EmbeddingDataset(
        X,
        y,
        test_indices
    ),
    batch_size=64,
    shuffle=False
)

In [14]:
test_embeddings = []
test_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        _, embeddings = model(images)

        embeddings = F.normalize(
            embeddings,
            dim=1
        )

        test_embeddings.append(
            embeddings.cpu().numpy()
        )

        test_labels.append(
            labels.numpy()
        )

test_embeddings = np.concatenate(
    test_embeddings
)

test_labels = np.concatenate(
    test_labels
)

print(test_embeddings.shape)

(9000, 128)


In [15]:
centroids_by_k = {}

for k in [1, 5, 10]:

    data = np.load(
        CENTROID_DIR
        / f"objects_{k}.npz"
    )

    centroids_by_k[k] = {
        name: data[name]
        for name in data.files
    }

In [16]:
import pandas as pd

summary_rows = []

for k in [1, 5, 10]:

    correct = 0

    true_scores = []
    false_scores = []
    gaps = []

    for embedding, label in zip(
        test_embeddings,
        test_labels
    ):

        true_class = categories[label]

        scores = {}

        for class_name in categories:

            centroid = (
                centroids_by_k[k][
                    class_name
                ][0]
            )

            scores[class_name] = float(
                centroid @ embedding
            )

        predicted = max(
            scores,
            key=scores.get
        )

        true_score = scores[
            true_class
        ]

        highest_false = max(
            score
            for name, score in scores.items()
            if name != true_class
        )

        gap = (
            true_score
            - highest_false
        )

        correct += (
            predicted == true_class
        )

        true_scores.append(
            true_score
        )

        false_scores.append(
            highest_false
        )

        gaps.append(
            gap
        )

    gaps = np.array(
        gaps
    )

    summary_rows.append({
        "k": k,

        "accuracy":
            correct / len(test_labels),

        "avg_true_score":
            np.mean(true_scores),

        "avg_highest_false":
            np.mean(false_scores),

        "avg_gap":
            np.mean(gaps),

        "margin_>=_0.10":
            np.mean(gaps >= 0.10)
    })

quickdraw_summary = pd.DataFrame(
    summary_rows
)

display(
    quickdraw_summary.round(4)
)

,k,accuracy,avg_true_score,avg_highest_false,avg_gap,margin_>=_0.10
0,1,0.8166,0.9038,0.7785,0.1253,0.6196
1,5,0.8221,0.9091,0.7797,0.1295,0.6247
2,10,0.8218,0.9097,0.7802,0.1295,0.6270


In [17]:
BASE_DRAWINGS_DIR = Path(
    "../../data/base_drawings/objects"
)

def preprocess_manual_image(
    image_path,
    padding=10
):

    image = Image.open(
        image_path
    ).convert("L")

    array = (
        255.0
        - np.array(
            image,
            dtype=np.float32
        )
    )

    mask = array > 20

    if mask.any():

        rows, cols = np.where(
            mask
        )

        top = max(
            rows.min() - padding,
            0
        )

        bottom = min(
            rows.max() + padding + 1,
            array.shape[0]
        )

        left = max(
            cols.min() - padding,
            0
        )

        right = min(
            cols.max() + padding + 1,
            array.shape[1]
        )

        array = array[
            top:bottom,
            left:right
        ]

    image = Image.fromarray(
        array.astype(np.uint8)
    )

    side = max(
        image.size
    )

    canvas = Image.new(
        "L",
        (side, side),
        0
    )

    x = (
        side - image.width
    ) // 2

    y = (
        side - image.height
    ) // 2

    canvas.paste(
        image,
        (x, y)
    )

    canvas = canvas.resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.LANCZOS
    )

    return (
        np.array(
            canvas,
            dtype=np.float32
        )
        / 255.0
    )

In [18]:
k = 5

rows = []
drawing_names = []

model.eval()

for true_class in categories:

    for quality in [
        "good",
        "bad"
    ]:

        path = (
            BASE_DRAWINGS_DIR
            / f"{true_class}_{quality}.jpg"
        )

        image = preprocess_manual_image(
            path
        )

        tensor = (
            torch.from_numpy(image)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        with torch.no_grad():

            _, embedding = model(
                tensor
            )

        embedding = (
            F.normalize(
                embedding,
                dim=1
            )
            .cpu()
            .numpy()[0]
        )

        scores = []

        for class_name in categories:

            centroid = (
                centroids_by_k[k][
                    class_name
                ][0]
            )

            scores.append(
                float(
                    centroid @ embedding
                )
            )

        rows.append(
            scores
        )

        drawing_names.append(
            f"{true_class}_{quality}"
        )


manual_similarity_table = pd.DataFrame(
    rows,
    index=drawing_names,
    columns=categories
)

display(
    manual_similarity_table.round(3)
)

,backpack,bed,book,chair,clock,cup,door,key,knife,laptop,shoe,spoon,table,television,toothbrush
backpack_good,0.657,0.710,0.839,0.615,0.448,0.722,0.582,0.592,0.559,0.822,0.677,0.534,0.599,0.864,0.539
backpack_bad,0.961,0.584,0.660,0.487,0.680,0.662,0.479,0.554,0.449,0.545,0.559,0.461,0.529,0.648,0.462
bed_good,0.749,0.762,0.912,0.647,0.507,0.821,0.668,0.666,0.625,0.686,0.677,0.627,0.675,0.721,0.648
bed_bad,0.621,0.790,0.709,0.772,0.435,0.666,0.588,0.616,0.667,0.701,0.630,0.598,0.643,0.731,0.616
book_good,0.767,0.848,0.860,0.708,0.541,0.859,0.667,0.701,0.661,0.705,0.722,0.647,0.724,0.741,0.678
book_bad,0.731,0.694,0.942,0.572,0.497,0.764,0.612,0.593,0.533,0.647,0.636,0.522,0.617,0.710,0.517
chair_good,0.672,0.801,0.735,0.867,0.494,0.781,0.704,0.684,0.705,0.695,0.701,0.664,0.699,0.648,0.693
chair_bad,0.494,0.611,0.548,0.813,0.366,0.586,0.531,0.634,0.805,0.567,0.560,0.739,0.516,0.519,0.778
clock_good,0.937,0.718,0.684,0.585,0.677,0.714,0.527,0.600,0.526,0.583,0.609,0.521,0.589,0.585,0.548
clock_bad,0.945,0.686,0.677,0.576,0.694,0.719,0.540,0.600,0.528,0.586,0.618,0.520,0.589,0.582,0.537
